# Лабораторна робота №2 — Частина 1
## Наука про дані: підготовчий етап
### VHI-індекс для адміністративних одиниць України

## 1. Імпорт бібліотек

In [7]:
import urllib.request
import pandas as pd
import os
import glob
import re
from datetime import datetime
from io import StringIO

## 2. Завантаження VHI-даних для всіх областей України

Завантажуємо дані VHI-індексу з NOAA для кожної з 27 адміністративних одиниць (provinceID від 1 до 27).
При повторному запуску файли не перезавантажуються — реалізовано захист від дублювання.

In [8]:
# Папка для збереження даних
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

def download_vhi_data(province_id, year1=1981, year2=2024):
    """
    Завантажує VHI-дані для вказаної області.
    Якщо дані вже завантажені — пропускає.
    До імені файлу додається дата та час завантаження.
    """
    # Перевіряємо чи файл для цієї області вже існує
    existing = glob.glob(os.path.join(DATA_DIR, f"vhi_province_{province_id}_*.csv"))
    if existing:
        print(f"  Область {province_id}: вже завантажено ({os.path.basename(existing[0])}), пропускаємо")
        return existing[0]
    
    url = (f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?"
           f"country=UKR&provinceID={province_id}&year1={year1}&year2={year2}&type=Mean")
    
    now = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = os.path.join(DATA_DIR, f"vhi_province_{province_id}_{now}.csv")
    
    try:
        urllib.request.urlretrieve(url, filename)
        print(f"  Область {province_id}: завантажено -> {os.path.basename(filename)}")
        return filename
    except Exception as e:
        print(f"  Область {province_id}: ПОМИЛКА - {e}")
        return None

# Завантажуємо дані для всіх 27 областей (provinceID=0 — середнє по Україні, не завантажуємо)
print("Завантаження VHI-даних для областей України...")
print("=" * 60)
downloaded_files = {}
for pid in range(1, 28):
    filepath = download_vhi_data(pid)
    if filepath:
        downloaded_files[pid] = filepath

print("=" * 60)
print(f"Завантажено файлів: {len(downloaded_files)}")

Завантаження VHI-даних для областей України...
  Область 1: вже завантажено (vhi_province_1_2026-04-03_00-47-06.csv), пропускаємо
  Область 2: вже завантажено (vhi_province_2_2026-04-03_00-47-13.csv), пропускаємо
  Область 3: вже завантажено (vhi_province_3_2026-04-03_00-47-13.csv), пропускаємо
  Область 4: вже завантажено (vhi_province_4_2026-04-03_00-47-14.csv), пропускаємо
  Область 5: вже завантажено (vhi_province_5_2026-04-03_00-47-15.csv), пропускаємо
  Область 6: вже завантажено (vhi_province_6_2026-04-03_00-47-16.csv), пропускаємо
  Область 7: вже завантажено (vhi_province_7_2026-04-03_00-47-17.csv), пропускаємо
  Область 8: вже завантажено (vhi_province_8_2026-04-03_00-47-18.csv), пропускаємо
  Область 9: вже завантажено (vhi_province_9_2026-04-03_00-47-19.csv), пропускаємо
  Область 10: вже завантажено (vhi_province_10_2026-04-03_00-47-20.csv), пропускаємо
  Область 11: вже завантажено (vhi_province_11_2026-04-03_00-47-21.csv), пропускаємо
  Область 12: вже завантажено (vhi_p

## 3. Зчитування та очищення даних (Data Cleaning)

Зчитуємо всі завантажені файли у pandas DataFrame. 
Файли NOAA мають формат: HTML-обгортка `<pre>...</pre>`, всередині — CSV з роздільником `,` та зайвою комою в кінці.
Прибираємо HTML-теги, зайві стовпці, заповнюємо пропуски. Додаємо стовпчики з назвою та індексом області.

In [9]:
# Маппінг NOAA індексів (англійська абетка) -> назва області
NOAA_ID_TO_NAME = {
    1: "Cherkasy", 2: "Chernihiv", 3: "Chernivtsi", 4: "Crimea",
    5: "Dnipropetrovsk", 6: "Donetsk", 7: "Ivano-Frankivsk",
    8: "Kharkiv", 9: "Kherson", 10: "Khmelnytskyi",
    11: "Kyiv", 12: "Kyiv City", 13: "Kirovohrad", 14: "Luhansk",
    15: "Lviv", 16: "Mykolaiv", 17: "Odessa", 18: "Poltava",
    19: "Rivne", 20: "Sevastopol", 21: "Sumy", 22: "Ternopil",
    23: "Transcarpathia", 24: "Vinnytsia", 25: "Volyn",
    26: "Zaporizhzhia", 27: "Zhytomyr"
}

# Маппінг: українська абетка -> NOAA ID
# Індексація за українською абеткою: 1-Вінницька, 2-Волинська, ...
UKR_INDEX = {
    1: ("Вінницька", 24),
    2: ("Волинська", 25),
    3: ("Дніпропетровська", 5),
    4: ("Донецька", 6),
    5: ("Житомирська", 27),
    6: ("Закарпатська", 23),
    7: ("Запорізька", 26),
    8: ("Івано-Франківська", 7),
    9: ("Київська", 11),
    10: ("Кіровоградська", 13),
    11: ("Кримська", 4),
    12: ("Луганська", 14),
    13: ("Львівська", 15),
    14: ("Миколаївська", 16),
    15: ("Одеська", 17),
    16: ("Полтавська", 18),
    17: ("Рівненська", 19),
    18: ("Сумська", 21),
    19: ("Тернопільська", 22),
    20: ("Харківська", 8),
    21: ("Херсонська", 9),
    22: ("Хмельницька", 10),
    23: ("Черкаська", 1),
    24: ("Чернівецька", 3),
    25: ("Чернігівська", 2),
    26: ("м. Київ", 12),
    27: ("м. Севастополь", 20),
}

# Зворотній маппінг: NOAA ID -> український індекс
NOAA_TO_UKR = {noaa_id: (ukr_idx, name) for ukr_idx, (name, noaa_id) in UKR_INDEX.items()}

print("Маппінг NOAA -> Українська індексація:")
print("-" * 50)
for noaa_id in sorted(NOAA_TO_UKR.keys()):
    ukr_idx, ukr_name = NOAA_TO_UKR[noaa_id]
    print(f"  NOAA {noaa_id:2d} ({NOAA_ID_TO_NAME[noaa_id]:20s}) -> {ukr_idx:2d} {ukr_name}")

Маппінг NOAA -> Українська індексація:
--------------------------------------------------
  NOAA  1 (Cherkasy            ) -> 23 Черкаська
  NOAA  2 (Chernihiv           ) -> 25 Чернігівська
  NOAA  3 (Chernivtsi          ) -> 24 Чернівецька
  NOAA  4 (Crimea              ) -> 11 Кримська
  NOAA  5 (Dnipropetrovsk      ) ->  3 Дніпропетровська
  NOAA  6 (Donetsk             ) ->  4 Донецька
  NOAA  7 (Ivano-Frankivsk     ) ->  8 Івано-Франківська
  NOAA  8 (Kharkiv             ) -> 20 Харківська
  NOAA  9 (Kherson             ) -> 21 Херсонська
  NOAA 10 (Khmelnytskyi        ) -> 22 Хмельницька
  NOAA 11 (Kyiv                ) ->  9 Київська
  NOAA 12 (Kyiv City           ) -> 26 м. Київ
  NOAA 13 (Kirovohrad          ) -> 10 Кіровоградська
  NOAA 14 (Luhansk             ) -> 12 Луганська
  NOAA 15 (Lviv                ) -> 13 Львівська
  NOAA 16 (Mykolaiv            ) -> 14 Миколаївська
  NOAA 17 (Odessa              ) -> 15 Одеська
  NOAA 18 (Poltava             ) -> 16 Полтавська
  

In [10]:
def read_vhi_file(filepath, noaa_province_id):
    """
    Зчитує один VHI-файл з NOAA та очищує дані.
    
    Формат файлу NOAA:
    - Обгорнутий в <pre>...</pre> теги
    - Заголовок: " Year, Week, SMN, SMT, VCI, TCI, VHI,"  (з зайвою комою)
    - Дані: " 1982,  1, 0.054, 262.29, 46.83, 31.75, 39.29,"
    - Між блоками років може бути порожній рядок
    """
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        
        # Видаляємо HTML-теги
        content = content.replace('<pre>', '').replace('</pre>', '')
        content = re.sub(r'<[^>]+>', '', content)
        
        # Розбиваємо на рядки та фільтруємо
        lines = content.strip().split('\n')
        
        # Формуємо чисті рядки даних
        header = "Year,Week,SMN,SMT,VCI,TCI,VHI"
        data_lines = [header]
        
        for line in lines:
            line = line.strip()
            if not line:
                continue
            # Пропускаємо рядок-заголовок (містить "Year")
            if 'Year' in line or 'Week' in line:
                continue
            # Пропускаємо текстові рядки (дата завантаження тощо)
            if any(c.isalpha() for c in line.replace(',', '').strip()):
                continue
            
            # Видаляємо зайву кому в кінці
            line = line.rstrip(',').strip()
            
            # Розбиваємо за комою і беремо тільки перші 7 значень
            parts = [p.strip() for p in line.split(',')]
            if len(parts) >= 7:
                data_lines.append(','.join(parts[:7]))
        
        # Створюємо DataFrame
        csv_text = '\n'.join(data_lines)
        df = pd.read_csv(StringIO(csv_text))
        
        # Конвертуємо в числа
        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Замінюємо -1 на NaN (пропуски в даних NOAA)
        df = df.replace(-1, pd.NA)
        df = df.replace(-1.0, pd.NA)
        
        # Видаляємо повністю порожні рядки
        df = df.dropna(how='all')
        
        # Видаляємо рядки де Year або Week пропущені
        df = df.dropna(subset=['Year', 'Week'])
        
        # Конвертуємо Year та Week в int
        df['Year'] = df['Year'].astype(int)
        df['Week'] = df['Week'].astype(int)
        
        # Додаємо стовпці з індексом та назвою області
        ukr_idx, ukr_name = NOAA_TO_UKR.get(noaa_province_id, (noaa_province_id, "Unknown"))
        df['NOAA_ID'] = noaa_province_id
        df['Province_ID'] = ukr_idx
        df['Province_Name'] = ukr_name
        
        return df
    
    except Exception as e:
        print(f"  Помилка при читанні {filepath}: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()

# Зчитуємо всі файли
print("Зчитування та очищення даних...")
print("=" * 60)

all_data = []
for noaa_id, filepath in sorted(downloaded_files.items()):
    df = read_vhi_file(filepath, noaa_id)
    if not df.empty:
        all_data.append(df)
        print(f"  NOAA {noaa_id:2d}: {len(df)} записів, стовпці: {list(df.columns)}")
    else:
        print(f"  NOAA {noaa_id:2d}: ПОМИЛКА ЧИТАННЯ")

# Об'єднуємо в один DataFrame
vhi_df = pd.concat(all_data, ignore_index=True)

print("=" * 60)
print(f"Загальний DataFrame: {vhi_df.shape[0]} рядків, {vhi_df.shape[1]} стовпців")
print(f"Стовпці: {list(vhi_df.columns)}")

Зчитування та очищення даних...
  NOAA  1: 2236 записів, стовпці: ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'NOAA_ID', 'Province_ID', 'Province_Name']
  NOAA  2: 2236 записів, стовпці: ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'NOAA_ID', 'Province_ID', 'Province_Name']
  NOAA  3: 2236 записів, стовпці: ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'NOAA_ID', 'Province_ID', 'Province_Name']
  NOAA  4: 2236 записів, стовпці: ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'NOAA_ID', 'Province_ID', 'Province_Name']
  NOAA  5: 2236 записів, стовпці: ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'NOAA_ID', 'Province_ID', 'Province_Name']
  NOAA  6: 2236 записів, стовпці: ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'NOAA_ID', 'Province_ID', 'Province_Name']
  NOAA  7: 2236 записів, стовпці: ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'NOAA_ID', 'Province_ID', 'Province_Name']
  NOAA  8: 2236 записів, стовпці: ['Year', 'Week', 'SMN', 'SMT', 

In [11]:
# Перегляд перших записів
print("Перші 10 записів:")
vhi_df.head(10)

Перші 10 записів:


,Year,Week,SMN,SMT,VCI,TCI,VHI,NOAA_ID,Province_ID,Province_Name
0,1982,1,0.053,260.31,45.01,39.46,42.23,1,23,Черкаська
1,1982,2,0.054,262.29,46.83,31.75,39.29,1,23,Черкаська
2,1982,3,0.055,263.82,48.13,27.24,37.68,1,23,Черкаська
3,1982,4,0.053,265.33,46.09,23.91,35.0,1,23,Черкаська
4,1982,5,0.05,265.66,41.46,26.65,34.06,1,23,Черкаська
5,1982,6,0.048,266.55,36.56,29.46,33.01,1,23,Черкаська
6,1982,7,0.048,267.84,32.17,31.14,31.65,1,23,Черкаська
7,1982,8,0.05,269.3,30.3,32.5,31.4,1,23,Черкаська
8,1982,9,0.052,270.75,28.23,35.22,31.73,1,23,Черкаська
9,1982,10,0.056,272.73,25.25,37.63,31.44,1,23,Черкаська


In [12]:
# Інформація про DataFrame
print("Інформація про DataFrame:")
print("=" * 60)
vhi_df.info()
print("\nСтатистика:")
vhi_df.describe()

Інформація про DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 60372 entries, 0 to 60371
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Year           60372 non-null  int64 
 1   Week           60372 non-null  int64 
 2   SMN            59022 non-null  object
 3   SMT            59022 non-null  object
 4   VCI            59022 non-null  object
 5   TCI            59022 non-null  object
 6   VHI            59022 non-null  object
 7   NOAA_ID        60372 non-null  int64 
 8   Province_ID    60372 non-null  int64 
 9   Province_Name  60372 non-null  str   
dtypes: int64(4), object(5), str(1)
memory usage: 4.6+ MB

Статистика:


,Year,Week,NOAA_ID,Province_ID
count,60372.000000,60372.000000,60372.000000,60372.000000
mean,2003.000000,26.500000,14.000000,14.000000
std,12.409776,15.008455,7.788945,7.788945
min,1982.000000,1.000000,1.000000,1.000000
25%,1992.000000,13.750000,7.000000,7.000000
50%,2003.000000,26.500000,14.000000,14.000000
75%,2014.000000,39.250000,21.000000,21.000000
max,2024.000000,52.000000,27.000000,27.000000


In [13]:
# Перевірка пропусків
print("Кількість пропусків по стовпцях:")
print(vhi_df.isnull().sum())
print(f"\nЗагальний відсоток пропусків: {vhi_df.isnull().sum().sum() / vhi_df.size * 100:.2f}%")

Кількість пропусків по стовпцях:
Year                0
Week                0
SMN              1350
SMT              1350
VCI              1350
TCI              1350
VHI              1350
NOAA_ID             0
Province_ID         0
Province_Name       0
dtype: int64

Загальний відсоток пропусків: 1.12%


## 4. Зміна індексів областей

В завантажених даних NOAA області індексуються за англійською абеткою (Province 1 - Cherkasy). 
Замінюємо на індексацію за українською абеткою (1 - Вінницька, 2 - Волинська, ...).

In [14]:
# Перевіримо що індекси правильно замінені
print("Унікальні області в DataFrame (українська індексація):")
print("=" * 60)
provinces = vhi_df[['Province_ID', 'Province_Name', 'NOAA_ID']].drop_duplicates().sort_values('Province_ID')
for _, row in provinces.iterrows():
    print(f"  {int(row['Province_ID']):2d}. {row['Province_Name']:25s} (NOAA ID: {int(row['NOAA_ID'])})")

print(f"\nВсього областей: {vhi_df['Province_ID'].nunique()}")

Унікальні області в DataFrame (українська індексація):
   1. Вінницька                 (NOAA ID: 24)
   2. Волинська                 (NOAA ID: 25)
   3. Дніпропетровська          (NOAA ID: 5)
   4. Донецька                  (NOAA ID: 6)
   5. Житомирська               (NOAA ID: 27)
   6. Закарпатська              (NOAA ID: 23)
   7. Запорізька                (NOAA ID: 26)
   8. Івано-Франківська         (NOAA ID: 7)
   9. Київська                  (NOAA ID: 11)
  10. Кіровоградська            (NOAA ID: 13)
  11. Кримська                  (NOAA ID: 4)
  12. Луганська                 (NOAA ID: 14)
  13. Львівська                 (NOAA ID: 15)
  14. Миколаївська              (NOAA ID: 16)
  15. Одеська                   (NOAA ID: 17)
  16. Полтавська                (NOAA ID: 18)
  17. Рівненська                (NOAA ID: 19)
  18. Сумська                   (NOAA ID: 21)
  19. Тернопільська             (NOAA ID: 22)
  20. Харківська                (NOAA ID: 8)
  21. Херсонська              

## 5. Функції для формування вибірок

### 5.1. Ряд VHI для області за вказаний рік

In [15]:
def get_vhi_for_province_year(df, province_id, year):
    """
    Повертає ряд VHI для вказаної області за вказаний рік.
    province_id — індекс за українською абеткою.
    """
    result = df[(df['Province_ID'] == province_id) & (df['Year'] == year)]
    
    if result.empty:
        province_name = UKR_INDEX.get(province_id, ("Невідома",))[0]
        print(f"Даних не знайдено для області {province_id} ({province_name}) за {year} рік")
        return result
    
    province_name = result['Province_Name'].iloc[0]
    print(f"VHI для {province_name} (індекс {province_id}) за {year} рік:")
    print(f"Кількість записів (тижнів): {len(result)}")
    return result[['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'Province_Name']].reset_index(drop=True)

# Приклад: VHI для Київської області (індекс 9) за 2020 рік
get_vhi_for_province_year(vhi_df, 9, 2020)

VHI для Київська (індекс 9) за 2020 рік:
Кількість записів (тижнів): 52


,Year,Week,SMN,SMT,VCI,TCI,VHI,Province_Name
0,2020,1,0.087,267.51,62.45,13.11,37.78,Київська
1,2020,2,0.085,266.49,62.2,14.63,38.41,Київська
2,2020,3,0.086,266.17,63.29,16.22,39.74,Київська
3,2020,4,0.088,266.04,64.06,19.75,41.9,Київська
4,2020,5,0.092,267.03,64.91,22.16,43.53,Київська
5,2020,6,0.1,268.92,64.8,21.95,43.37,Київська
6,2020,7,0.107,270.97,63.0,20.04,41.5,Київська
7,2020,8,0.119,273.95,63.93,15.17,39.53,Київська
8,2020,9,0.138,277.29,68.0,9.86,38.9,Київська
9,2020,10,0.156,280.11,70.3,7.9,39.07,Київська


### 5.2. Ряд VHI за вказаний діапазон років для вказаних областей

In [16]:
def get_vhi_for_provinces_years(df, province_ids, year_start, year_end):
    """
    Повертає VHI за діапазон років для кількох областей.
    province_ids — список індексів за українською абеткою.
    """
    result = df[
        (df['Province_ID'].isin(province_ids)) & 
        (df['Year'] >= year_start) & 
        (df['Year'] <= year_end)
    ]
    
    if result.empty:
        print(f"Даних не знайдено для областей {province_ids} за {year_start}-{year_end}")
        return result
    
    provinces_found = result['Province_Name'].unique()
    print(f"VHI за {year_start}-{year_end} для областей: {', '.join(provinces_found)}")
    print(f"Кількість записів: {len(result)}")
    return result[['Year', 'Week', 'VHI', 'Province_ID', 'Province_Name']].reset_index(drop=True)

# Приклад: VHI для Львівської (13) та Одеської (15) областей за 2015-2020
get_vhi_for_provinces_years(vhi_df, [13, 15], 2015, 2020)

VHI за 2015-2020 для областей: Львівська, Одеська
Кількість записів: 624


,Year,Week,VHI,Province_ID,Province_Name
0,2015,1,44.89,13,Львівська
1,2015,2,46.2,13,Львівська
2,2015,3,47.29,13,Львівська
3,2015,4,46.65,13,Львівська
4,2015,5,46.38,13,Львівська
...,...,...,...,...,...
619,2020,48,44.29,15,Одеська
620,2020,49,45.8,15,Одеська
621,2020,50,44.82,15,Одеська
622,2020,51,45.14,15,Одеська


### 5.3. Пошук екстремумів (min, max), середнього та медіани VHI

In [17]:
def get_vhi_extremes(df, province_ids, year_start, year_end):
    """
    Знаходить min, max, середнє та медіану VHI для вказаних областей та років.
    """
    result = df[
        (df['Province_ID'].isin(province_ids)) & 
        (df['Year'] >= year_start) & 
        (df['Year'] <= year_end)
    ].copy()
    
    if result.empty:
        print("Даних не знайдено")
        return pd.DataFrame()
    
    print(f"Екстремуми VHI за {year_start}-{year_end}")
    print("=" * 70)
    
    stats_list = []
    for pid in province_ids:
        prov_data = result[result['Province_ID'] == pid]['VHI'].dropna()
        if prov_data.empty:
            continue
        
        name = UKR_INDEX.get(pid, ("Невідома",))[0]
        stats = {
            'Область': name,
            'ID': pid,
            'Min VHI': prov_data.min(),
            'Max VHI': prov_data.max(),
            'Середнє': round(prov_data.mean(), 2),
            'Медіана': round(prov_data.median(), 2),
            'Кількість': len(prov_data)
        }
        stats_list.append(stats)
        
        # Знаходимо тиждень/рік для min та max
        min_row = result[(result['Province_ID'] == pid) & (result['VHI'] == prov_data.min())].iloc[0]
        max_row = result[(result['Province_ID'] == pid) & (result['VHI'] == prov_data.max())].iloc[0]
        
        print(f"  {name} (ID {pid}):")
        print(f"    Min VHI = {prov_data.min():.2f} (рік {int(min_row['Year'])}, тиждень {int(min_row['Week'])})")
        print(f"    Max VHI = {prov_data.max():.2f} (рік {int(max_row['Year'])}, тиждень {int(max_row['Week'])})")
        print(f"    Середнє = {prov_data.mean():.2f}, Медіана = {prov_data.median():.2f}")
        print()
    
    return pd.DataFrame(stats_list)

# Приклад: екстремуми для Харківської (20), Дніпропетровської (3) за 2000-2020
get_vhi_extremes(vhi_df, [20, 3], 2000, 2020)

Екстремуми VHI за 2000-2020
  Харківська (ID 20):
    Min VHI = 9.36 (рік 2000, тиждень 48)
    Max VHI = 91.42 (рік 2003, тиждень 32)
    Середнє = 48.69, Медіана = 48.02

  Дніпропетровська (ID 3):
    Min VHI = 17.77 (рік 2000, тиждень 50)
    Max VHI = 90.29 (рік 2003, тиждень 31)
    Середнє = 46.47, Медіана = 45.44



,Область,ID,Min VHI,Max VHI,Середнє,Медіана,Кількість
0,Харківська,20,9.36,91.42,48.69,48.02,1072
1,Дніпропетровська,3,17.77,90.29,46.47,45.44,1072


### 5.4. Додаткові вибірки

#### 5.4.1. Роки з екстремальною посухою (VHI < 15) для вказаної області

In [18]:
def get_drought_years(df, province_id, threshold=15):
    """
    Знаходить роки, коли VHI опускався нижче порогового значення (екстремальна посуха).
    """
    prov_data = df[df['Province_ID'] == province_id].copy()
    drought = prov_data[prov_data['VHI'] < threshold]
    
    name = UKR_INDEX.get(province_id, ("Невідома",))[0]
    
    if drought.empty:
        print(f"Для {name} (ID {province_id}) не знайдено тижнів з VHI < {threshold}")
        return pd.DataFrame()
    
    drought_years = drought['Year'].unique()
    print(f"Роки з екстремальною посухою (VHI < {threshold}) для {name}:")
    print(f"  {sorted([int(y) for y in drought_years if pd.notna(y)])}")
    print(f"  Всього тижнів з VHI < {threshold}: {len(drought)}")
    
    return drought[['Year', 'Week', 'VHI', 'Province_Name']].sort_values(['Year', 'Week']).reset_index(drop=True)

# Приклад: посухи для Херсонської області (21)
get_drought_years(vhi_df, 21)

Роки з екстремальною посухою (VHI < 15) для Херсонська:
  [2003, 2007]
  Всього тижнів з VHI < 15: 12


,Year,Week,VHI,Province_Name
0,2003,20,14.53,Херсонська
1,2007,23,12.41,Херсонська
2,2007,24,12.23,Херсонська
3,2007,25,12.99,Херсонська
4,2007,26,13.33,Херсонська
5,2007,27,12.88,Херсонська
6,2007,28,12.63,Херсонська
7,2007,29,12.96,Херсонська
8,2007,30,13.48,Херсонська
9,2007,31,14.05,Херсонська


#### 5.4.2. Порівняння середнього VHI між областями за рік

In [19]:
def compare_provinces_vhi(df, year):
    """
    Порівнює середнє VHI всіх областей за вказаний рік.
    Повертає відсортований DataFrame.
    """
    year_data = df[df['Year'] == year]
    
    if year_data.empty:
        print(f"Даних за {year} рік не знайдено")
        return pd.DataFrame()
    
    comparison = year_data.groupby(['Province_ID', 'Province_Name'])['VHI'].agg(
        ['mean', 'min', 'max', 'count']
    ).round(2).reset_index()
    
    comparison.columns = ['ID', 'Область', 'Середнє VHI', 'Min VHI', 'Max VHI', 'Тижнів']
    comparison = comparison.sort_values('Середнє VHI', ascending=False).reset_index(drop=True)
    
    print(f"Порівняння областей за середнім VHI у {year} році:")
    return comparison

# Приклад: порівняння за 2020 рік
compare_provinces_vhi(vhi_df, 2020)

Порівняння областей за середнім VHI у 2020 році:


,ID,Область,Середнє VHI,Min VHI,Max VHI,Тижнів
0,6,Закарпатська,52.085385,38.94,68.67,52
1,24,Чернівецька,51.249231,42.25,66.4,52
2,19,Тернопільська,51.248846,40.2,68.55,52
3,2,Волинська,51.245962,39.48,61.93,52
4,17,Рівненська,50.839423,37.68,66.01,52
5,13,Львівська,50.625962,41.35,63.06,52
6,8,Івано-Франківська,50.169423,39.78,62.91,52
7,22,Хмельницька,49.138462,37.43,67.65,52
8,1,Вінницька,45.911538,34.48,64.12,52
9,18,Сумська,44.558269,24.52,72.11,52


#### 5.4.3. Тренд VHI для області по роках (середнє за рік)

In [20]:
def get_vhi_trend(df, province_id):
    """
    Показує тренд середнього VHI по роках для вказаної області.
    """
    prov_data = df[df['Province_ID'] == province_id]
    name = UKR_INDEX.get(province_id, ("Невідома",))[0]
    
    if prov_data.empty:
        print(f"Даних для {name} не знайдено")
        return pd.DataFrame()
    
    trend = prov_data.groupby('Year')['VHI'].mean().round(2).reset_index()
    trend.columns = ['Рік', 'Середнє VHI']
    
    print(f"Тренд VHI для {name} (ID {province_id}):")
    return trend

# Приклад: тренд для Полтавської області (16)
get_vhi_trend(vhi_df, 16)

Тренд VHI для Полтавська (ID 16):


,Рік,Середнє VHI
0,1982,48.23
1,1983,39.74
2,1984,44.28
3,1985,46.0
4,1986,33.6
5,1987,53.8
6,1988,53.23
7,1989,39.18
8,1990,50.64
9,1991,46.39
